# 02c_deconfound_v2 — khử artifact theo TẦNG, đo bằng bộ đặc trưng khoá cứng

**Vị trí trong pipeline** — chạy THAY CHO `02b_deconfound`:

    02_assemble -> [attack-dataset/paraphrase/llm-attack] -> 02c_deconfound_v2
    -> 03_diagnostics -> 04_melt -> train

**Input** : `output/dataset_{train,val,test_ood_model,test_fp}_normalized.parquet`
(+ `*_perturbed.parquet` nếu đã chạy attack — tự phát hiện)
**Output**: `output/dataset_{split}_deconf2.parquet` (bản lọc, dùng để train/eval)
và `output/dataset_{split}_deconf2_all.parquet` (đủ dòng + cờ, dùng cho ablation).
KHÔNG ghi đè `_normalized` lẫn `_deconf`.

## Vì sao có bản v2 khi đã có 02b

02b tự chấm điểm bằng SÁU đặc trưng, và bộ sáu đó không chứa `avg_word_len` lẫn
`type_token_ratio` — đúng hai đặc trưng bề mặt mạnh nhất. Đo lại trên cùng file
`dataset_train_deconf.parquet` mà 02b sinh ra:

| bộ đặc trưng dùng để đo | AUC | kết luận |
|---|---:|---|
| 6 đặc trưng (02b tự chấm) | 0,6867 | "G6 PASS" |
| 22 đặc trưng (`shortcut_audit.py`) | **0,9114** | WARN |

Con số PASS đó là hệ quả của việc CHỌN đặc trưng. Ngoài ra 02b còn để lại ba chỗ
chưa xử lý, đều đo được:

1. `truncate_pair` cắt thô theo từ -> tỉ lệ văn bản kết thúc đúng dấu câu tụt
   **99,0% -> 52,4%**. Cân bằng ở cả hai lớp nên không thành shortcut, nhưng nửa
   tập train kết thúc giữa câu trong khi inference gặp văn bản nguyên vẹn.
2. Ba kênh trung thực-nội dung chưa từng được xét: `single_char_frac` 0,624 ·
   `titlecase_token_frac` 0,620 · `numeric_token_frac` 0,575. human_text là đoạn
   bài báo thật nên còn biến toán, số liệu, tên riêng; bản AI viết lại lược đi.
3. `char_count` 0,717 — 02b cân bằng số TỪ nên số KÝ TỰ vẫn lệch 15%.
   (Đây là đặc trưng dẫn xuất: char_count = word_count x avg_word_len, nên nó tự
   hết khi word_count được kiểm soát và avg_word_len được công nhận là văn phong.)

## v2 làm gì

| bước | mục đích | phạm vi |
|---|---|---|
| `canonicalize` | phiên chuyển non-ASCII, chuẩn hoá dấu câu | train VÀ inference |
| `truncate_pair_sentences` | cân bằng số từ, cắt ở BIÊN CÂU | chỉ dựng dataset |
| lọc `notation_gap` | bỏ cặp mà bản AI lược ký hiệu/số/tên riêng nhiều nhất | chỉ dựng dataset |

Bước 3 là bước MỚI và nó là bước lọc CẶP, không phải phép biến đổi văn bản: không
thể xoá số khỏi human_text mà không xoá nội dung. Nó cũng là tiêu chí CHẤT LƯỢNG
sinh dữ liệu (bản viết lại phải giữ được số liệu và tên riêng của bài gốc), nên bỏ
những cặp đó đúng theo cả hai nghĩa.

Mọi phép biến đổi nằm trong `preprocess.py`, mọi phép đo nằm trong
`shortcut_audit.py`. Notebook này chỉ điều phối.

In [ ]:
# ── CELL 1: Config ──────────────────────────────────────────────
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "preprocess.py").exists()), Path.cwd())
sys.path.insert(0, str(BASE_DIR))

from preprocess import (canonicalize, non_ascii_coverage, notation_gap,  # noqa: E402
                        notation_profile, truncate_pair_sentences, NOTATION_GAP_KEYS)
import shortcut_audit as SA  # noqa: E402

OUTPUT_DIR = BASE_DIR / "output"

SPLIT_SOURCES = {
    "train":          ["dataset_train_normalized.parquet"],
    "val":            ["dataset_val_perturbed.parquet", "dataset_val_normalized.parquet"],
    "test_ood_model": ["dataset_test_ood_model_perturbed.parquet",
                       "dataset_test_ood_model_normalized.parquet"],
    "test_fp":        ["dataset_test_fp_normalized.parquet"],
}
ATTACK_COLS = ["ai_text_deepwordbug", "ai_text_textbugger",
               "ai_text_paraphrase", "ai_text_llm_mixing"]

# Split áp được cắt-cặp và lọc-cặp. test_fp không có cặp thật (ai_text null ở
# 3.335/4.290 dòng vì nó đo dương tính giả) nên CHỈ canonicalize. Phải ghi rõ điều
# này trong KLTN: chuẩn hoá giữa các split không đồng nhất ở bước cắt.
PAIRED_SPLITS = {"train", "val", "test_ood_model"}

# Tỉ lệ cặp GIỮ LẠI sau lọc notation_gap. Ngưỡng tuyệt đối được lấy từ phân vị này
# TRÊN TRAIN rồi áp cho val/test — không lấy phân vị riêng từng split, vì như vậy là
# để tiêu chí dựng dữ liệu phụ thuộc vào phân phối của chính tập test.
#
# Chọn 0,85 bằng số đo, không bằng cảm giác (audit đầy đủ trên train):
#     keep 100% : fidelity 0,7993  single_char 0,622  titlecase 0,619  uppercase 0,655
#     keep  90% : fidelity 0,7768                                       uppercase 0,628
#     keep  85% : fidelity 0,7666  single_char 0,588  titlecase 0,579  uppercase 0,617
#     keep  80% : fidelity 0,7564                                       uppercase 0,608
# Từ 85% xuống 80% mua thêm 0,009 bằng 5% dữ liệu -> không đáng.
NOTATION_KEEP_FRAC = 0.85

# Hạ chữ toàn bộ văn bản ở CẢ HAI lớp. Tắt mặc định — đây là một đánh đổi, không phải
# một cải tiến, và số đo cho thấy nó không mua được gì đáng kể:
#
#     deconf2 cased    : TỔNG 0,8909 | fidelity 0,7666 | uppercase_ratio 0,617 -> WARN
#     deconf2 lowercase: TỔNG 0,8895 | fidelity 0,7074 | (không kênh nào vượt) -> PASS
#
# Bật lên thì ba kênh case (uppercase_ratio, titlecase_token_frac, allcaps_token_frac)
# về 0,5 và audit PASS cả hai cổng. Nhưng TỔNG chỉ nhích 0,0014 — nghĩa là mô hình
# nông vốn đã bão hoà bằng kênh văn phong, ba kênh case gần như không góp gì. Đổi lại,
# detector mất hẳn thông tin viết hoa, mà đó là thông tin THẬT và có ở mọi văn bản
# lúc inference.
#
# Nếu bật: PHẢI hạ chữ cả ở inference (thêm vào canonicalize), nếu không thì lệch
# train/inference. Và phải chạy lại toàn bộ model.
LOWERCASE = False

print(f"BASE_DIR : {BASE_DIR}")
print(f"LOWERCASE: {LOWERCASE}")
print(f"lọc notation: giữ {NOTATION_KEEP_FRAC:.0%} cặp, ngưỡng lấy từ TRAIN")
print(f"kênh vào notation_gap: {', '.join(NOTATION_GAP_KEYS)}")

resolved = {}
for split, candidates in SPLIT_SOURCES.items():
    for c in candidates:
        if (OUTPUT_DIR / c).exists():
            resolved[split] = OUTPUT_DIR / c
            break
    print(f"  {split:<16}: {'OK ' + resolved[split].name if split in resolved else 'KHONG TIM THAY'}")
assert "train" in resolved, "Thiếu train -> không tính được ngưỡng lọc"

In [ ]:
# ── CELL 2: QA bảng phiên chuyển — ký tự nào bị bỏ sót? ─────────
# Ký tự không có trong NON_ASCII_TABLE và NFKD cũng không cứu được sẽ thành khoảng
# trắng, tức MẤT NGHĨA. Nếu danh sách dưới đây có ký hiệu toán phổ biến thì phải bổ
# sung vào preprocess.NON_ASCII_TABLE rồi chạy lại notebook.
import unicodedata  # noqa: E402

print("=" * 78)
print("QA — độ phủ của bảng phiên chuyển (trên train)")
print("=" * 78)

_raw_train = pd.read_parquet(resolved["train"])
total, mapped, leftover = non_ascii_coverage(
    pd.concat([_raw_train["human_text"], _raw_train["ai_text"]]).tolist())
print(f"  ký tự non-ASCII  : {total:,}")
print(f"  phiên chuyển được: {mapped:,}  ({100 * mapped / max(total, 1):.2f}%)")
print(f"  bị bỏ sót        : {total - mapped:,}  ({100 * (total - mapped) / max(total, 1):.2f}%)")
for c, k in leftover:
    try:
        nm = unicodedata.name(c)
    except ValueError:
        nm = "?"
    print(f"    {c!r:>10} x{k:<7} U+{ord(c):04X}  {nm}")

In [ ]:
# ── CELL 3: canonicalize -> truncate_pair_sentences ─────────────
def n_words(text):
    return 0 if text is None or pd.isna(text) else len(str(text).split())


def transform(df, split):
    """canonicalize mọi cột text, rồi cắt cặp theo biên câu (split có cặp)."""
    out = df.copy()
    text_cols = [c for c in ["human_text", "ai_text", *ATTACK_COLS] if c in out.columns]

    for col in text_cols:
        out[col] = [None if pd.isna(t) else canonicalize(t) for t in out[col]]
        if LOWERCASE:
            out[col] = [None if t is None else t.lower() for t in out[col]]

    if split in PAIRED_SPLITS:
        pairs = [truncate_pair_sentences(h, a)
                 for h, a in zip(out["human_text"], out["ai_text"])]
        out["human_text"] = [p[0] for p in pairs]
        out["ai_text"] = [p[1] for p in pairs]

        # Biến thể attack cắt về CÙNG số từ với ai_text, không sinh lại attack.
        # Cắt theo TỪ ở đây là đúng: bản attack là ai_text đã bị nhiễu ở mức ký tự
        # nên biên câu của nó không còn đáng tin, còn số từ thì vẫn tương ứng 1-1.
        L = [n_words(t) for t in out["ai_text"]]
        for col in ATTACK_COLS:
            if col in out.columns:
                out[col] = [None if pd.isna(t) else " ".join(str(t).split()[:k])
                            for t, k in zip(out[col], L)]
    else:
        print(f"    {split}: không có cặp -> chỉ canonicalize, KHÔNG cắt, KHÔNG lọc")

    if "word_count" in out.columns:
        out["word_count"] = [n_words(t) for t in out["ai_text"]]
    return out, text_cols


print("=" * 78)
print("BƯỚC 1-2: canonicalize -> truncate_pair_sentences")
print("=" * 78)

staged = {}
for split, path in resolved.items():
    df = pd.read_parquet(path)
    print(f"\n  {split} ({len(df):,} dòng) <- {path.name}")
    new, text_cols = transform(df, split)

    before = sum(df[c].map(n_words).sum() for c in text_cols)
    after = sum(new[c].map(n_words).sum() for c in text_cols)
    print(f"    cột text: {', '.join(text_cols)}")
    print(f"    số từ: {before:,} -> {after:,}  (giữ {100 * after / max(before, 1):.1f}%)")

    if split in PAIRED_SPLITS:
        hw = new["human_text"].map(n_words)
        aw = new["ai_text"].map(n_words)
        delta = (aw - hw).abs()
        term_h = new["human_text"].dropna().str.rstrip().str.endswith((".", "!", "?")).mean()
        term_a = new["ai_text"].dropna().str.rstrip().str.endswith((".", "!", "?")).mean()
        print(f"    số từ sau cắt: human {hw.mean():.1f} | AI {aw.mean():.1f}"
              f"  (lệch trung bình {aw.mean() - hw.mean():+.2f} từ)")
        print(f"    |chênh| trong cặp: median {delta.median():.0f} · p90 {delta.quantile(.9):.0f}"
              f" · ngoài dung sai {100 * (delta > 8).mean():.1f}%")
        print(f"    kết thúc đúng dấu câu: human {term_h:.1%} | AI {term_a:.1%}"
              "   (02b: 54,8% / 50,1%)")

    for c in text_cols:
        empty = new[c].notna() & (new[c].map(n_words) == 0)
        if empty.any():
            print(f"    CANH BAO {c}: {int(empty.sum())} dòng RỖNG sau biến đổi")

    staged[split] = new

In [ ]:
# ── CELL 4: BƯỚC 3 — lọc cặp theo notation_gap ──────────────────
print("=" * 78)
print("BƯỚC 3: lọc cặp theo notation_gap (ngưỡng lấy từ TRAIN)")
print("=" * 78)


def gap_series(df):
    """notation_gap cho từng dòng. NaN ở dòng thiếu một bên (không lọc được)."""
    out = []
    for h, a in zip(df["human_text"], df["ai_text"]):
        if h is None or a is None or pd.isna(h) or pd.isna(a):
            out.append(np.nan)
        else:
            out.append(notation_gap(notation_profile(h), notation_profile(a)))
    return pd.Series(out, index=df.index, dtype=float)

gaps = {sp: gap_series(df) for sp, df in staged.items()}
NOTATION_GAP_THRESHOLD = float(gaps["train"].quantile(NOTATION_KEEP_FRAC))
print(f"\n  ngưỡng tuyệt đối (phân vị {NOTATION_KEEP_FRAC:.0%} của train)"
      f" = {NOTATION_GAP_THRESHOLD:.4f}")
print(f"  phân phối gap trên train: median {gaps['train'].median():.4f}"
      f" · p90 {gaps['train'].quantile(.90):.4f} · p99 {gaps['train'].quantile(.99):.4f}")

deconf2, deconf2_all = {}, {}
print(f"\n  {'split':<16}{'dòng':>8}{'giữ':>8}{'tỉ lệ':>9}  ghi chú")
for split, df in staged.items():
    g = gaps[split]
    full = df.copy()
    full["notation_gap"] = g.values
    # NaN (thiếu một bên) -> GIỮ: không lọc được thì không loại, và test_fp toàn NaN.
    ok = ~(g > NOTATION_GAP_THRESHOLD)
    if split not in PAIRED_SPLITS:
        ok = pd.Series(True, index=df.index)
    full["notation_ok"] = ok.values
    deconf2_all[split] = full
    deconf2[split] = full[ok.values].reset_index(drop=True)
    note = "chỉ canonicalize" if split not in PAIRED_SPLITS else ""
    print(f"  {split:<16}{len(df):>8,}{len(deconf2[split]):>8,}"
          f"{len(deconf2[split]) / len(df):>9.1%}  {note}")

In [ ]:
# ── CELL 5: VERIFY — audit đầy đủ, không tự chọn đặc trưng ──────
# Dùng shortcut_audit.py, tức ĐÚNG bộ đặc trưng và ĐÚNG ngưỡng mà 03_diagnostics và
# mọi báo cáo dùng. Không khai lại FEATURE_COLS ở đây — đó là cách 02b đi chệch.
print("=" * 78)
print("VERIFY — audit 4 tầng trên train, so ba biến thể")
print("=" * 78)

_variants = {
    "normalized (gốc)": pd.read_parquet(resolved["train"]),
    "deconf   (02b)": (OUTPUT_DIR / "dataset_train_deconf.parquet"),
    "deconf2  (02c)": deconf2["train"],
}
_summary = []
for name, obj in _variants.items():
    if isinstance(obj, Path):
        if not obj.exists():
            print(f"[skip] {name}: {obj.name} không tồn tại")
            continue
        obj = pd.read_parquet(obj)
    res = SA.audit_frame(obj.dropna(subset=["human_text", "ai_text"]), name)
    SA.print_audit(res)
    t = res["totals"]
    _summary.append({
        "variant": name, "n_pairs": res["n_rows"] // 2,
        "total": t["all|GBM"][0], "process": t["process_only|GBM"][0],
        "fidelity": t["fidelity_only|GBM"][0], "style": t["style_only|GBM"][0],
        "legacy_6feat": t["legacy_g6_6feat|GBM"][0],
        "worst_process": f"{res['worst_process']['feature']}={res['worst_process']['auc_mean']:.4f}",
        "worst_fidelity": f"{res['worst_fidelity']['feature']}={res['worst_fidelity']['auc_mean']:.4f}",
        "verdict": res["verdict"],
    })

_sum_df = pd.DataFrame(_summary)
print("BẢNG SO SÁNH BA BIẾN THỂ (đây là bảng đưa vào KLTN)")
print(_sum_df.to_string(index=False))

In [ ]:
# ── CELL 6: Ghi output ─────────────────────────────────────────
print("=" * 78)
print("GHI OUTPUT (không ghi đè _normalized lẫn _deconf)")
print("=" * 78)

written = []
for split, df in deconf2.items():
    p = OUTPUT_DIR / f"dataset_{split}_deconf2.parquet"
    df.to_parquet(p, index=False)
    written.append(p)
    print(f"  {p.name:<46}{len(df):>7,} dòng")
for split, df in deconf2_all.items():
    p = OUTPUT_DIR / f"dataset_{split}_deconf2_all.parquet"
    df.to_parquet(p, index=False)
    print(f"  {p.name:<46}{len(df):>7,} dòng  (đủ dòng + cờ notation_ok)")

_meta = {
    "built_at": pd.Timestamp.now().isoformat(timespec="seconds"),
    "recipe": ["canonicalize", *(["lowercase"] if LOWERCASE else []),
               "truncate_pair_sentences", "notation_gap filter"],
    "lowercase": LOWERCASE,
    "notation_gap_keys": list(NOTATION_GAP_KEYS),
    "notation_keep_frac": NOTATION_KEEP_FRAC,
    "notation_gap_threshold": NOTATION_GAP_THRESHOLD,
    "sources": {k: v.name for k, v in resolved.items()},
    "rows_kept": {k: int(len(v)) for k, v in deconf2.items()},
    "rows_total": {k: int(len(v)) for k, v in deconf2_all.items()},
    "audit": _summary,
}
(OUTPUT_DIR / "deconf2_manifest.json").write_text(
    json.dumps(_meta, indent=2, ensure_ascii=False), encoding="utf-8")
_sum_df.to_csv(OUTPUT_DIR / "deconf2_variant_comparison.csv", index=False)
print(f"\n  deconf2_manifest.json + deconf2_variant_comparison.csv")

# Bảng long của 04 vừa bị làm CŨ. Không cảnh báo thì bạn train trên bảng long của lần
# chạy trước mà không có gì báo lỗi — đúng loại bug im lặng tệ nhất.
newest = max(p.stat().st_mtime for p in written)
stale = [q for split in deconf2
         if (q := OUTPUT_DIR / f"dataset_{split}_final_long_deconf2.parquet").exists()
         and q.stat().st_mtime < newest]
if stale:
    print(f"\n  CANH BAO: {len(stale)} bảng long đã CŨ hơn dữ liệu vừa ghi:")
    for q in stale:
        print(f"       {q.name}")

print("""
BƯỚC TIẾP
  1. pipeline_config.py : DATASET_VARIANT = "deconf2"
  2. 03_diagnostics.ipynb : cổng G6 giờ đọc shortcut_audit -> process phải PASS
  3. 04_melt_for_training.ipynb : chạy lại để sinh bảng long deconf2
  4. make_matched_eval.py : dựng tập eval matched (surface AUC ~0,5)
  5. surface_baseline.py  : sàn bề mặt chính thức cho leaderboard
  6. retrain trên Kaggle -> so OOD AUROC với 0,9898 (deberta-v3 bản deconf)
  7. INFERENCE: chỉ `canonicalize`. KHÔNG áp truncate_pair_sentences (cần cặp) và
     KHÔNG áp lọc notation (là tiêu chí dựng dữ liệu, không phải tiền xử lý).
""")